# 6.8 · 主成分分析 / Principal Component Analysis (PCA)

> **课程定位 / Where this fits**
> 前 7 课讲聚类(发现分组), 这一课转入**降维**(发现紧凑表示)。PCA 是最经典、最重要的线性降维: 找数据中**方差最大的正交方向(主成分)**, 用最少维度保留最多信息。它无处不在——可视化、去噪、压缩、加速下游模型、缓解维度灾难(5.3)。5.12 见过 LDA, 这里把 PCA 讲透。
> PCA finds orthogonal directions of maximum variance to compress data into few dimensions while keeping the most information. The foundational linear dimensionality reduction.

> 💡 **面试相关 / Interview-relevant**
> - "PCA 在做什么 / 主成分是什么" ★★★★★
> - "PCA 与协方差矩阵特征分解 / SVD 的关系" ★★★★★
> - "为什么 PCA 前要中心化(和标准化)" ★★★★★
> - "怎么选主成分个数(解释方差)" ★★★★
> - "PCA vs LDA" ★★★★（无监督 vs 有监督）
> - "PCA 的假设和局限(线性/正交/方差≠重要)" ★★★★

---

## 学习目标 / Learning Objectives
1. PCA 的两种等价视角: 最大方差 / 最小重构误差。
2. 用协方差特征分解 / SVD 从零实现。
3. 中心化与标准化的必要性。
4. 解释方差比选维度 + 重构/去噪。
5. PCA 的假设与局限(引出核 PCA 6.9)。

## 目录 / TOC
1. [两种视角 + 数学 ⭐](#1)
2. [🌸 数据: Iris + 从零(SVD) ⭐](#2)
3. [中心化/标准化 ⭐](#3)
4. [解释方差选维度 ⭐](#4)
5. [🔢 重构与去噪: Digits](#5)
6. [假设与局限](#6)
7. [小结](#7)


<a id="1"></a>
## 1. 两种视角 + 数学 ⭐ / Two Views & Math

PCA 有两个等价的定义(都导出同一答案):
- **最大方差**: 找一个方向 $\mathbf{w}$($\|\mathbf{w}\|=1$)使投影 $\mathbf{Xw}$ 的方差最大。第二主成分在与第一正交的约束下方差最大, 以此类推。
- **最小重构误差**: 找一个 $k$ 维子空间, 使所有点到它的投影误差(平方和)最小。

**数学解**: 设数据已中心化(每列减均值), 协方差矩阵 $\mathbf{C}=\frac1n\mathbf{X}^\top\mathbf{X}$。最大化 $\mathbf{w}^\top\mathbf{C}\mathbf{w}$ s.t. $\|\mathbf{w}\|=1$, 用拉格朗日得:
$$\mathbf{C}\mathbf{w} = \lambda\mathbf{w}$$
即**主成分 = 协方差矩阵的特征向量**, 对应特征值 $\lambda$ = 该方向上的方差。按 $\lambda$ 从大到小取前 $k$ 个特征向量即可。

**SVD 视角**(数值上更稳, sklearn 用它): $\mathbf{X}=\mathbf{U}\boldsymbol\Sigma\mathbf{V}^\top$, 则 $\mathbf{V}$ 的列就是主成分, 奇异值平方 $\propto$ 特征值。


<a id="2"></a>
## 2. 数据: Iris + 从零(SVD) ⭐ / Iris & From Scratch

先用 **Iris**(4 维 → 2 维)做可视化, 从零用 SVD 实现并对照 sklearn。


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
sns.set_theme(style="whitegrid")
np.set_printoptions(precision=3, suppress=True)

iris = load_iris(); X, y = iris.data, iris.target
Xc = X - X.mean(0)                      # 中心化(PCA 必须!)

# 从零: SVD
U, S, Vt = np.linalg.svd(Xc, full_matrices=False)
PC = Vt[:2]                             # 前两个主成分(行向量)
Z_scratch = Xc @ PC.T                   # 投影到 2 维
explained = (S**2) / (S**2).sum()       # 解释方差比

from sklearn.decomposition import PCA
pca = PCA(n_components=2).fit(Xc)
Z_sk = pca.transform(Xc)
print(f"从零解释方差比(前2): {explained[:2]}  累计 {explained[:2].sum():.3f}")
print(f"sklearn 解释方差比:  {pca.explained_variance_ratio_}")
print(f"投影一致(符号可能相反): {np.allclose(np.abs(Z_scratch), np.abs(Z_sk), atol=1e-6)}")

fig, ax = plt.subplots(figsize=(6.5,5))
for k,name in enumerate(iris.target_names):
    ax.scatter(Z_scratch[y==k,0], Z_scratch[y==k,1], label=name, s=25)
ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.legend()
ax.set_title(f"Iris PCA 4D→2D: 前2主成分保留 {explained[:2].sum():.0%} 方差")
plt.tight_layout(); plt.show()


<a id="3"></a>
## 3. 中心化 / 标准化 ⭐ / Centering & Scaling

- **中心化(减均值)是必须的**: 协方差/方差的定义就基于偏离均值; 不中心化, 第一主成分会指向数据的"平均位置"而非变化方向。sklearn 的 PCA **自动中心化**。
- **标准化(再除标准差)视情况**: PCA 看方差, 若特征量纲悬殊(收入 vs 年龄), 大量纲特征会主导主成分。量纲不一时应先标准化(等价于用**相关矩阵**而非协方差矩阵做 PCA)。


In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_wine
wine = load_wine()
Xw = wine.data
# 不标准化: 某些大量纲特征(如 proline ~1000)主导
pca_raw = PCA(2).fit(Xw - Xw.mean(0))
pca_std = PCA(2).fit(StandardScaler().fit_transform(Xw))
print("Wine 数据(特征量纲差异大, 如 proline~1000 vs flavanoids~3):")
print(f"  不标准化 PC1 解释方差比: {pca_raw.explained_variance_ratio_[0]:.3f} (被大量纲特征绑架)")
print(f"  标准化后 PC1 解释方差比: {pca_std.explained_variance_ratio_[0]:.3f} (更均衡反映各特征)")
print("\n量纲悬殊时先标准化(=用相关矩阵); 同量纲(如像素)可只中心化")


<a id="4"></a>
## 4. 解释方差选维度 ⭐ / Choosing Components

每个主成分的**解释方差比** = $\lambda_k / \sum\lambda$。常用做法: 画**累计解释方差**曲线(scree plot), 取"保留 90%/95% 方差"所需的维度, 或找肘部。


In [ ]:
pca_full = PCA().fit(StandardScaler().fit_transform(Xw))
cum = np.cumsum(pca_full.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(range(1, len(cum)+1), pca_full.explained_variance_ratio_, alpha=0.5, label="单个")
ax.plot(range(1, len(cum)+1), cum, "o-", color="red", label="累计")
ax.axhline(0.95, color="gray", ls="--", label="95% 阈值")
k95 = np.argmax(cum >= 0.95) + 1
ax.axvline(k95, color="green", ls=":", label=f"{k95} 维达 95%")
ax.set_xlabel("主成分数"); ax.set_ylabel("解释方差比"); ax.legend()
ax.set_title(f"Scree plot: Wine 13 维 → {k95} 维即保留 95% 方差")
plt.tight_layout(); plt.show()
print(f"Wine: {k95} 个主成分保留 95% 方差(从 13 维压缩); sklearn 可直接 PCA(n_components=0.95)")


<a id="5"></a>
## 5. 重构与去噪: Digits / Reconstruction & Denoising

PCA 可**逆变换**(近似重构): 用前 k 个主成分重建数据, 丢掉的是小方差方向——常常是噪声。这给了 PCA 一个**去噪/压缩**用途。用 **Digits**(64 维像素)演示。


In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()
Xd = digits.data
rng = np.random.default_rng(0)
Xn = Xd + rng.normal(0, 4, Xd.shape)     # 加噪声

pca_d = PCA(n_components=20).fit(Xd)      # 64→20 维
Xden = pca_d.inverse_transform(pca_d.transform(Xn))   # 投影再重构=去噪
print(f"Digits 64维 → 20 主成分保留方差: {pca_d.explained_variance_ratio_.sum():.2%}")

fig, axes = plt.subplots(3, 8, figsize=(11, 4.2))
for j in range(8):
    axes[0,j].imshow(Xd[j].reshape(8,8), cmap="gray_r"); axes[0,j].axis("off")
    axes[1,j].imshow(Xn[j].reshape(8,8), cmap="gray_r"); axes[1,j].axis("off")
    axes[2,j].imshow(Xden[j].reshape(8,8), cmap="gray_r"); axes[2,j].axis("off")
axes[0,0].set_ylabel("原始", rotation=0);
for ax,t in zip(axes[:,0], ["原始","加噪","PCA去噪"]): ax.set_title("")
fig.text(0.09, 0.78, "原始", va="center"); fig.text(0.09, 0.5, "加噪", va="center"); fig.text(0.09, 0.22, "去噪", va="center")
plt.suptitle("PCA 去噪: 投影到前20主成分再重构, 丢掉的小方差方向多是噪声")
plt.tight_layout(); plt.show()


<a id="6"></a>
## 6. 假设与局限 / Assumptions & Limitations

- **线性**: PCA 只找线性子空间。数据在**弯曲流形**上(瑞士卷)时, PCA 无能为力 → 核 PCA(6.9)/t-SNE(6.12)/UMAP(6.13)。
- **方差 ≠ 重要性**: PCA 假设大方差方向更重要, 但有时判别信息藏在小方差方向里(此时 LDA 5.12 更合适——它用标签找可分方向)。
- **正交 + 全局**: 成分必须正交, 且是全局线性组合, 可解释性有限。
- **对异常值敏感**(方差被离群点拉偏)。


<a id="7"></a>
## 7. 小结 / Summary

```
PCA: 找方差最大的正交方向(主成分)= 协方差矩阵的特征向量(或 X 的 SVD 右奇异向量)
两视角等价: 最大化投影方差 = 最小化重构误差
中心化必须(sklearn 自动); 量纲悬殊先标准化(=相关矩阵 PCA)
选维度: 累计解释方差(95%) / scree plot 肘部; PCA(n_components=0.95)
用途: 可视化/压缩/去噪(逆变换丢小方差方向)/加速下游/缓解维度灾难
局限: 线性、方差≠重要、正交全局、对异常敏感 → 非线性用核PCA/t-SNE/UMAP
```

### 💡 面试速查
1. **主成分=协方差矩阵特征向量**(按特征值降序); 特征值=该方向方差
2. **SVD** 数值更稳(sklearn 用); 最大方差 ⟺ 最小重构误差
3. **必须中心化**; 量纲不一**先标准化**
4. **选维度**看累计解释方差(常 90/95%)
5. **PCA(无监督,方差) vs LDA(有监督,可分性)**; 局限是线性 → 核PCA

### 下一节
**6.9 核 PCA**——用核技巧(5.5 见过)把 PCA 推广到非线性: 在隐式高维空间做 PCA, 能展开瑞士卷这类弯曲流形。
